# K-Means Clustering Data Polutan Udara

## Daftar isi
1. Python
2. KNIME

# 1. Python

Alur pengolahan data pada bagian Python:

1. **Ambil data.** Dua tabel fitur TSFEL (skenario interpolasi linear dan polinomial) dibaca dari database MySQL.
2. **Pisahkan fitur.** Kolom `id`, `nama`, dan `daerah` dipisahkan dari 204 kolom fitur numerik.
3. **Standardisasi.** Fitur distandardisasi dengan `StandardScaler` (Z-Score) agar tiap fitur memiliki skala yang sebanding.
4. **Reduksi dimensi.** Dibuat empat kondisi: 204 fitur asli, lalu 203, 74, dan 37 fitur memakai `FeatureAgglomeration` (metode Ward).
5. **K-Means.** Untuk tiap kondisi, K-Means diuji pada K = 2 sampai 10, dengan Silhouette Score dan inertia (Elbow) sebagai pembanding.
6. **Hasil.** K final mengikuti Silhouette Score, lalu anggota tiap cluster ditampilkan dalam tabel dan dipetakan berdasarkan koordinat daerah.

Seluruh proses dijalankan untuk dua skenario dan empat kondisi dimensi, sehingga menghasilkan **delapan kombinasi** hasil.

## 1.1 Library dan Koneksi Database

Bagian ini menyiapkan seluruh pustaka dan membuka koneksi ke database MySQL tempat dua tabel fitur (skenario linear dan polinomial) disimpan. Koneksi dipakai ulang di seluruh sel berikutnya, jadi cukup dijalankan sekali di awal.

> **Password database tidak ditulis di notebook.** Simpan sebagai Colab Secret bernama `DB_PASS` (atau variabel lingkungan `DB_PASS`). Bila tidak ada, notebook meminta input saat dijalankan. Bila database tidak bisa dijangkau, notebook memakai berkas CSV `ekstraksi_fitur_linear.csv` dan `ekstraksi_fitur_polynomial.csv` di folder kerja (ekspor tabelnya).

In [ ]:
import subprocess, sys
for mod, pkg in [("pymysql", "pymysql"), ("sqlalchemy", "sqlalchemy"), ("geopy", "geopy"), ("folium", "folium")]:
    try: __import__(mod)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

DB_HOST = "basisdata2-c.my.id"
DB_PORT = 3306
DB_NAME = "basisda1_PSD-A-Interpolasi"
DB_USER = "basisda1_PSD-User"

def ambil_password():
    try:
        from google.colab import userdata
        return userdata.get("DB_PASS")
    except Exception:
        pass
    pw = os.environ.get("DB_PASS")
    if pw: return pw
    from getpass import getpass
    return getpass("Password database (kosongkan untuk memakai CSV): ")

BASE_DIR = Path(os.getcwd())
engine = None
try:
    pw = ambil_password()
    if pw:
        # Password mengandung karakter "#", sehingga perlu di-encode dengan quote_plus
        # sebelum disisipkan ke URL koneksi.
        db_url = (f"mysql+pymysql://{quote_plus(DB_USER)}:{quote_plus(pw)}"
                  f"@{DB_HOST}:{DB_PORT}/{DB_NAME}")
        engine = create_engine(db_url, connect_args={"connect_timeout": 15})
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("Koneksi database berhasil.")
except Exception as e:
    engine = None
    print("Koneksi database gagal -> memakai berkas CSV bila ada.\n  ", str(e)[:150])

# K-Means diuji untuk k = 2 sampai 10.
K_RANGE = range(2, 11)
RANDOM_STATE = 42
N_INIT = 10
MAX_ITER = 300

**Penjelasan kode:**
1. Koneksi dibuat dengan SQLAlchemy dan driver `pymysql`, karena database bertipe MySQL.
2. `quote_plus()` dipakai pada password supaya karakter `#` di dalamnya tidak mengacaukan struktur URL koneksi.
3. `K_RANGE` menetapkan rentang jumlah cluster yang diuji di sepanjang notebook, yaitu 2 sampai 10.

## 1.2 Memuat Data Dua Skenario Interpolasi

In [ ]:
TABEL = {
    "linear": "ekstraksi_fitur_linear",
    "polynomial": "ekstraksi_fitur_polynomial",
}

def muat_tabel(nama_tabel):
    if engine is not None:
        with engine.connect() as conn:
            return pd.read_sql(text(f"SELECT * FROM `{nama_tabel}` ORDER BY id;"), conn)
    csv = BASE_DIR / f"{nama_tabel}.csv"
    if csv.exists():
        return pd.read_csv(csv).sort_values("id").reset_index(drop=True)
    raise FileNotFoundError(f"Tidak bisa membaca '{nama_tabel}': database tidak terhubung dan {csv.name} tidak ada di {BASE_DIR}.")

data_mentah = {}
for skenario, nama_tabel in TABEL.items():
    df = muat_tabel(nama_tabel)
    data_mentah[skenario] = df
    print(f"Skenario {skenario}: {df.shape[0]} baris, {df.shape[1]} kolom")
    print(f"Nilai kosong: {int(df.isna().sum().sum())}")
    display(df.iloc[:, :8].head(10))
    print()

**Penjelasan kode:** kedua tabel dimuat ke dalam satu dictionary `data_mentah` dengan kunci nama skenarionya, supaya proses selanjutnya bisa diulang dengan cara yang sama untuk kedua skenario tanpa menulis kode dua kali.

## 1.3 Pemisahan Kolom Identitas dan Kolom Fitur

Tiap tabel punya tiga kolom identitas, yaitu `id`, `nama`, dan `daerah`, yang bukan bagian dari fitur polutan dan tidak diikutsertakan dalam perhitungan jarak antar data saat klustering. Sisanya (204 kolom) adalah fitur numerik hasil ekstraksi TSFEL untuk tiga polutan.

In [ ]:
KOLOM_IDENTITAS = ["id", "nama", "daerah"]
info_skenario = {}
for skenario, df in data_mentah.items():
    kolom_fitur = [c for c in df.columns if c not in KOLOM_IDENTITAS]
    X_mentah = df[kolom_fitur].apply(pd.to_numeric, errors="coerce")
    info_skenario[skenario] = {"df": df, "kolom_fitur": kolom_fitur, "X_mentah": X_mentah}
    print(f"Skenario {skenario}: {len(kolom_fitur)} kolom fitur, "
          f"{X_mentah.isna().sum().sum()} nilai kosong terdeteksi")

**Penjelasan kode:** `pd.to_numeric(errors="coerce")` memaksa setiap kolom fitur menjadi angka dan mengubah nilai yang tidak terbaca menjadi kosong (`NaN`), supaya ketahuan lebih dulu kalau ada kolom bermasalah sebelum masuk ke tahap standardisasi.

## 1.4 Fungsi Evaluasi K-Means, Silhouette, dan Elbow

In [ ]:
def evaluasi_kmeans(X, k_range=K_RANGE):
    hasil = []
    for k in k_range:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT, max_iter=MAX_ITER)
        label = model.fit_predict(X)
        hasil.append({"k": k, "silhouette": silhouette_score(X, label), "inertia": model.inertia_})
    return pd.DataFrame(hasil)

def k_terbaik_silhouette(tabel_k):
    baris = tabel_k.loc[tabel_k["silhouette"].idxmax()]
    return int(baris["k"]), float(baris["silhouette"])

# Dari grafik Elbow pada data ini, titik elbow dibaca (secara visual) pada K = 4.
# Nilai inertia tetap dihitung dan grafik tetap ditampilkan agar keputusan dapat diperiksa.
# Ubah K_ELBOW bila data berubah dan titik siku pada grafik bergeser.
K_ELBOW = 4

def k_terbaik_elbow(tabel_k):
    return K_ELBOW

**Penjelasan kode:** `evaluasi_kmeans()` mencoba setiap k dari 2 sampai 10, menjalankan K-Means untuk tiap k, lalu menghitung silhouette score dan inertia-nya. Semakin tinggi silhouette, semakin jelas pemisahan antar cluster. `k_terbaik_silhouette()` mengambil baris dengan silhouette tertinggi. Titik elbow tidak dihitung otomatis, melainkan dibaca dari grafik dan ditetapkan lewat `K_ELBOW`.

## 1.5 Reduksi Dimensi 203, 74, dan 37 Fitur

Tahap analisis menggunakan empat kondisi: tanpa reduksi (204 fitur asli), lalu reduksi ke 203, 74, dan 37 fitur.

PCA, cara reduksi dimensi yang paling umum, punya batas matematis: jumlah komponen tidak boleh melebihi `min(jumlah_sampel, jumlah_fitur)`. Karena jumlah daerah di database masih di bawah 203, PCA gagal untuk target 203 maupun 74. Karena itu dipakai **`FeatureAgglomeration`** sebagai gantinya untuk ketiga tahap reduksi.

**Konsep `FeatureAgglomeration` dibanding PCA**

PCA mencari kombinasi dari seluruh kolom yang menangkap variasi antar baris, sehingga jumlah komponennya dibatasi jumlah baris. `FeatureAgglomeration` justru mengelompokkan **kolom (fitur)**, bukan baris, dengan cara seperti hierarchical clustering:

1. Dihitung seberapa mirip tiap pasang kolom (jarak Euclidean antar kolom; tiap kolom dianggap satu titik di ruang berdimensi jumlah baris).
2. Dimulai dari 204 kolom terpisah, dua kolom paling mirip digabung menjadi satu kelompok.
3. Langkah ini diulang, menggabung dua kelompok paling mirip, sampai jumlah kelompok sesuai target (mis. 37).
4. Tiap kelompok diwakili satu nilai: rata-rata dari semua kolom anggotanya.

Karena yang dikelompokkan adalah kolom, jumlah kelompok yang diminta tidak dibatasi jumlah baris data, sehingga target 203, 74, dan 37 selalu bisa dicapai persis.

Konsekuensinya, kolom hasil adalah gabungan beberapa fitur asli (tidak bisa lagi disebut satu per satu), dan tidak ada ukuran "persentase variasi yang terjaga" sejelas `explained_variance_ratio_` pada PCA.

In [ ]:
from sklearn.cluster import FeatureAgglomeration

TAHAP_REDUKSI = [None, 203, 74, 37]

def reduksi_dimensi(X_scaled, n_target):
    if n_target is None:
        return X_scaled, X_scaled.shape[1], None
    reducer = FeatureAgglomeration(n_clusters=n_target, linkage="ward")
    X_reduksi = reducer.fit_transform(X_scaled)
    return X_reduksi, X_reduksi.shape[1], reducer

**Penjelasan kode:** `linkage="ward"` menentukan aturan menilai "seberapa mirip dua kelompok kolom" setiap kali digabung. Metode Ward dipilih karena cenderung menghasilkan kelompok berukuran seimbang. `n_clusters=n_target` memastikan jumlah kolom akhir tepat sesuai target (203, 74, atau 37). Kondisi tanpa reduksi memakai 204 fitur asli dan tidak melewati `FeatureAgglomeration`.

## 1.6 Eksperimen Tanpa Reduksi dan Dengan Reduksi Dimensi

In [ ]:
hasil_eksperimen = {}
ringkasan_list = []

for skenario, info in info_skenario.items():
    df = info["df"]
    X_mentah = info["X_mentah"]

    baris_valid = X_mentah.dropna().index
    df_valid = df.loc[baris_valid].reset_index(drop=True)
    X_valid = X_mentah.loc[baris_valid].reset_index(drop=True)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_valid)

    for dimensi_diminta in TAHAP_REDUKSI:
        dimensi = X_scaled.shape[1] if dimensi_diminta is None else dimensi_diminta
        X_tahap, dimensi_efektif, reducer = reduksi_dimensi(X_scaled, dimensi_diminta)

        tabel_k = evaluasi_kmeans(X_tahap)
        k_silhouette, silhouette_terbaik = k_terbaik_silhouette(tabel_k)
        k_elbow = k_terbaik_elbow(tabel_k)

        # K final mengikuti Silhouette Score.
        model_final = KMeans(n_clusters=k_silhouette, random_state=RANDOM_STATE, n_init=N_INIT, max_iter=MAX_ITER)
        label_cluster = model_final.fit_predict(X_tahap)

        df_hasil = df_valid[["nama", "daerah"]].copy()
        df_hasil["cluster"] = label_cluster

        kunci = (skenario, dimensi)
        hasil_eksperimen[kunci] = {
            "dimensi_efektif": dimensi_efektif, "tabel_k": tabel_k,
            "k_silhouette": k_silhouette, "k_elbow": k_elbow, "k_terbaik": k_silhouette,
            "silhouette_terbaik": silhouette_terbaik, "df_hasil": df_hasil,
            "X_tahap": X_tahap, "model_final": model_final, "reducer": reducer,
        }
        ringkasan_list.append({
            "skenario": skenario, "dimensi": dimensi, "dimensi_efektif": dimensi_efektif,
            "k_silhouette": k_silhouette, "k_elbow": k_elbow, "k_terbaik": k_silhouette,
            "silhouette_terbaik": round(silhouette_terbaik, 4),
            "inertia_k_terbaik": round(model_final.inertia_, 4),
        })
        print(f"{skenario} | {dimensi} fitur | K Silhouette={k_silhouette} | K Elbow={k_elbow} | "
              f"K final={k_silhouette} | Silhouette={silhouette_terbaik:.4f}")

**Penjelasan kode:** setiap skenario diuji pada empat kondisi (204, 203, 74, dan 37 fitur). Untuk tiap kondisi, K-Means diuji pada K = 2 sampai 10. Silhouette Score dipakai untuk mencari K tertinggi, dan inertia dipakai untuk grafik Elbow. **K final mengikuti hasil Silhouette.**

## 1.7 Penentuan K Terbaik

Merangkum hasil bagian 1.6 ke satu tabel: K yang dipilih lewat Silhouette Score dan K yang dibaca lewat Elbow untuk tiap kombinasi skenario dan dimensi.

In [ ]:
ringkasan = pd.DataFrame(ringkasan_list)
ringkasan_terurut = ringkasan.sort_values("silhouette_terbaik", ascending=False).reset_index(drop=True)
display(ringkasan_terurut)

### 1.7.1 Grafik Silhouette dan Elbow

In [ ]:
for skenario in TABEL.keys():
    dimensi_list = [204, 203, 74, 37]
    fig, axes = plt.subplots(4, 2, figsize=(15, 18))
    for i, dimensi in enumerate(dimensi_list):
        hasil = hasil_eksperimen[(skenario, dimensi)]
        tabel_k = hasil["tabel_k"]

        axes[i, 0].plot(tabel_k["k"], tabel_k["silhouette"], marker="o")
        axes[i, 0].axvline(hasil["k_silhouette"], linestyle="--", label=f"K terbaik = {hasil['k_silhouette']}")
        axes[i, 0].set_title(f"{skenario} | {dimensi} fitur | Silhouette")
        axes[i, 0].set_xlabel("Jumlah Cluster (K)"); axes[i, 0].set_ylabel("Silhouette Score")
        axes[i, 0].set_xticks(list(K_RANGE)); axes[i, 0].grid(alpha=0.3); axes[i, 0].legend()

        axes[i, 1].plot(tabel_k["k"], tabel_k["inertia"], marker="o")
        axes[i, 1].axvline(hasil["k_elbow"], linestyle="--", label=f"K elbow = {hasil['k_elbow']}")
        axes[i, 1].set_title(f"{skenario} | {dimensi} fitur | Elbow Method")
        axes[i, 1].set_xlabel("Jumlah Cluster (K)"); axes[i, 1].set_ylabel("Inertia")
        axes[i, 1].set_xticks(list(K_RANGE)); axes[i, 1].grid(alpha=0.3); axes[i, 1].legend()
    plt.tight_layout()
    plt.show()

**Penjelasan kode:** grafik kiri menunjukkan Silhouette Score untuk K = 2 sampai 10; nilai tertinggi menjadi K terbaik menurut Silhouette. Grafik kanan menunjukkan inertia; titik elbow dibaca dari grafik (di sini K = 4, sesuai `K_ELBOW`).

## 1.8 Perbandingan K Terbaik

Tabel ringkasan bagian 1.7 ditampilkan terpisah per skenario, supaya perubahan K dan silhouette antar tahap dimensi (204, 203, 74, 37) lebih mudah dibaca berurutan.

In [ ]:
for skenario in TABEL.keys():
    subset = ringkasan[ringkasan["skenario"] == skenario].sort_values("dimensi", ascending=False)
    display(subset[["skenario", "dimensi", "dimensi_efektif", "k_silhouette", "k_elbow", "k_terbaik", "silhouette_terbaik"]])

**Penjelasan kode:** `dimensi` adalah target reduksi yang diminta dan `dimensi_efektif` adalah jumlah kolom yang benar-benar dipakai. Karena memakai `FeatureAgglomeration`, keduanya sama; reduksi ke 203 dari 204 hanya menggabung satu pasang fitur sehingga hasilnya hampir identik dengan 204.

## 1.9 Distribusi Anggota Cluster

Sebelum memilih hasil akhir, dicek apakah pembagian cluster masuk akal dengan melihat berapa daerah di tiap cluster. Cluster yang anggotanya hanya satu atau dua daerah perlu dicurigai sebagai outlier, bukan pengelompokan bermakna.

In [ ]:
for (skenario, dimensi), hasil in hasil_eksperimen.items():
    print(f"\nSkenario {skenario}, dimensi {dimensi} (K final={hasil['k_terbaik']}, "
          f"K silhouette={hasil['k_silhouette']}, K elbow={hasil['k_elbow']}, "
          f"silhouette={hasil['silhouette_terbaik']:.4f})")
    distribusi = hasil["df_hasil"]["cluster"].value_counts().sort_index()
    display(distribusi.rename("jumlah_anggota"))

**Penjelasan kode:** untuk tiap kombinasi, kolom `cluster` dihitung dengan `value_counts()` sehingga terlihat berapa daerah masuk ke tiap cluster. Cluster berisi satu atau dua daerah ditandai sebagai kandidat outlier.

*Pada run di laporan asli, hasilnya konsisten: satu cluster besar (34–35 daerah) dan satu cluster kecil (2–3 daerah), kecuali Polynomial 37 fitur dengan K = 3 (35, 1, dan 1 daerah).*

## 1.10 Hasil Klustering untuk Seluruh Delapan Kombinasi

Ada delapan kombinasi (dua skenario × empat tahap dimensi). Bagian ini menampilkan hasil kedelapannya satu per satu, bukan hanya satu yang dianggap "terbaik", supaya semuanya bisa dibandingkan dan dilaporkan.

In [ ]:
DIMENSI_URUT = [204, 203, 74, 37]
hasil_per_kombinasi = {}

for skenario in TABEL.keys():
    for dimensi in DIMENSI_URUT:
        hasil = hasil_eksperimen[(skenario, dimensi)]
        hasil_per_kombinasi[(skenario, dimensi)] = hasil["df_hasil"].copy()
        print(f"=== Skenario: {skenario} | Dimensi diminta: {dimensi} | Dimensi efektif: {hasil['dimensi_efektif']} ===")
        print(f"K Silhouette = {hasil['k_silhouette']} | K Elbow = {hasil['k_elbow']} | "
              f"K final dipakai = {hasil['k_terbaik']} | Silhouette = {hasil['silhouette_terbaik']:.4f}")
        display(hasil["df_hasil"])
        print()

**Penjelasan kode:** hasil klustering tiap kombinasi disalin ke dictionary `hasil_per_kombinasi` dengan kunci `(skenario, dimensi)`, lalu ditampilkan bersama K Silhouette, K Elbow, K final, dan nilai Silhouette. Dictionary ini dipakai ulang pada pemetaan (1.12).

Label cluster (0, 1, 2) hanya penanda dan tidak punya arti urutan. Karena itu, hasil antar kombinasi dibandingkan lewat **siapa saja anggota tiap cluster**, bukan angka labelnya. Contohnya, pada Linear 74 fitur cluster kecil berlabel 0, sedangkan pada dimensi lain berlabel 1, padahal anggotanya sama.

## 1.11 Mengambil Koordinat Tiap Daerah

Nama daerah yang sama muncul di kedelapan kombinasi (hanya pengelompokan clusternya yang beda), jadi koordinat tiap daerah cukup dicari sekali di sini lalu dipakai ulang di 1.12. Hasil pencarian disimpan di `koordinat_daerah.csv` sebagai cache; hapus berkas itu bila ingin mencari ulang.

In [ ]:
import requests
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

CACHE_KOORDINAT = BASE_DIR / "koordinat_daerah.csv"

KOREKSI_DAERAH = {
    "Bandung - Jogoroto, Jombang": "Jogoroto, Jombang",
    "Banyu Ajuh, Perumnas, Kamal": "Kamal, Bangkalan",
    "Kec. Kalianget, Sumenep": "Kalianget, Sumenep",
}

semua_daerah = set()
for df_hasil in hasil_per_kombinasi.values():
    semua_daerah.update(df_hasil["daerah"].unique())

koordinat = {}
if CACHE_KOORDINAT.exists():
    cache = pd.read_csv(CACHE_KOORDINAT)
    koordinat = {r.daerah: (r.lat, r.lon) for r in cache.itertuples() if pd.notna(r.lat)}
    print(f"Cache koordinat dimuat: {len(koordinat)} daerah")

geolocator = Nominatim(user_agent="klustering_polutan_indonesia")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def cari_koordinat(nama_daerah):
    nama = KOREKSI_DAERAH.get(nama_daerah, nama_daerah)
    try:
        hasil = geocode(f"{nama}, Indonesia")
        if hasil is not None:
            return hasil.latitude, hasil.longitude
    except Exception:
        pass
    try:
        r = requests.get("https://photon.komoot.io/api/",
                         params={"q": f"{nama}, Indonesia", "limit": 1}, timeout=10)
        fitur = r.json().get("features", [])
        if fitur:
            lon, lat = fitur[0]["geometry"]["coordinates"]
            return lat, lon
    except Exception:
        pass
    return None, None

daerah_gagal = []
for nama_daerah in sorted(semua_daerah):
    if nama_daerah in koordinat:
        continue
    lat, lon = cari_koordinat(nama_daerah)
    if lat is None: daerah_gagal.append(nama_daerah)
    else: koordinat[nama_daerah] = (lat, lon)

koordinat_daerah = pd.DataFrame([{"daerah": d, "lat": v[0], "lon": v[1]} for d, v in koordinat.items()])
koordinat_daerah.to_csv(CACHE_KOORDINAT, index=False)

print(f"Berhasil dapat koordinat: {len(semua_daerah) - len(daerah_gagal)} dari {len(semua_daerah)} daerah")
if daerah_gagal:
    print("Daerah yang masih gagal:")
    for d in daerah_gagal: print(" -", d)

**Penjelasan kode:** `cari_koordinat()` mencari latitude dan longitude tiap daerah. Pencarian pertama memakai Nominatim (OpenStreetMap) dengan jeda satu detik per permintaan sesuai aturan layanannya; bila tidak ditemukan dipakai layanan cadangan `photon.komoot.io`. Kamus `KOREKSI_DAERAH` memperbaiki penulisan daerah yang tidak baku (mis. "Banyu Ajuh, Perumnas, Kamal" menjadi "Kamal, Bangkalan") sebelum dicari. Nama daerah dikumpulkan dalam `set` agar tiap nama hanya dicari sekali.

## 1.12 Visualisasi Hasil Klustering pada Peta Satelit, Seluruh Delapan Kombinasi

Delapan peta ditampilkan berurutan sesuai `DIMENSI_URUT`, satu peta untuk tiap kombinasi skenario dan dimensi, supaya pola pengelompokan bisa dibandingkan langsung, termasuk melihat apakah pengelompokan berubah banyak saat dimensi direduksi.

In [ ]:
import folium
from IPython.display import display as tampilkan

WARNA = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue", "darkgreen", "pink", "black"]

for skenario in TABEL.keys():
    for dimensi in DIMENSI_URUT:
        hasil = hasil_eksperimen[(skenario, dimensi)]
        df_hasil = hasil_per_kombinasi[(skenario, dimensi)]

        data_peta = df_hasil.merge(koordinat_daerah, on="daerah", how="left")
        data_peta = data_peta.dropna(subset=["lat", "lon"])

        print(f"=== Skenario: {skenario} | Dimensi: {dimensi} | K final = {hasil['k_terbaik']} | "
              f"Silhouette = {hasil['silhouette_terbaik']:.4f} ===")
        if len(data_peta) == 0:
            print("Tidak ada koordinat yang berhasil ditemukan untuk kombinasi ini.")
            continue

        peta = folium.Map(
            location=[data_peta["lat"].mean(), data_peta["lon"].mean()],
            zoom_start=6,
            tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
            attr="Esri World Imagery",
        )
        warna_cluster = {c: WARNA[i % len(WARNA)] for i, c in enumerate(sorted(data_peta["cluster"].unique()))}
        for _, baris in data_peta.iterrows():
            folium.CircleMarker(
                location=[baris["lat"], baris["lon"]], radius=7,
                color=warna_cluster[baris["cluster"]], fill=True,
                fill_color=warna_cluster[baris["cluster"]], fill_opacity=0.85,
                popup=f"{baris['daerah']} (cluster {baris['cluster']})",
            ).add_to(peta)
        tampilkan(peta)

**Penjelasan kode:** untuk tiap kombinasi, tabel hasil klustering digabung dengan koordinat daerah (`merge`), lalu tiap daerah digambar sebagai titik di peta satelit Esri memakai `folium.CircleMarker`. Warna titik menunjukkan cluster, dan popup menampilkan nama daerah serta label cluster. Pusat peta diambil dari rata-rata koordinat titik yang berhasil ditemukan.

---
# 2. KNIME

Implementasi clustering di KNIME mengikuti alur yang sama dengan bagian Python. Bedanya, di KNIME tiap skenario hanya memakai beberapa konfigurasi K-Means yang dipilih:

- **Linear:** 2 node `k-Means`, keduanya dengan K = 2. Satu tanpa reduksi dimensi (204 fitur), satu dengan reduksi dimensi (203, 74, atau 37 fitur).
- **Polynomial:** 3 node `k-Means`. Satu tanpa reduksi dengan K = 2, satu dengan reduksi 203 atau 74 fitur dengan K = 2, dan satu dengan reduksi 37 fitur dengan K = 3.

## 2.1 Alur Pengolahan Data

1. **Pembacaan data.** Data fitur TSFEL tiap skenario dibaca dari database lewat node `DB Reader` (atau pembaca tabel lain). Linear memakai tabel `ekstraksi_fitur_linear`, Polynomial memakai `ekstraksi_fitur_polynomial`.
2. **Tanpa reduksi dimensi.** Data langsung diteruskan ke node `k-Means`, sehingga seluruh 204 fitur dipakai. Kolom identitas (`id`, `nama`, `daerah`) tidak dipakai sebagai fitur.
3. **Dengan reduksi dimensi.** Data diteruskan ke node `Python Script` yang menjalankan `FeatureAgglomeration` (Ward) dengan target 203, 74, atau 37 fitur, sama seperti bagian Python. Hasilnya diteruskan ke `k-Means`.
4. **K-Means.** Tiap node `k-Means` dijalankan dengan jumlah cluster yang ditetapkan untuk skenarionya (lihat 2.2 dan 2.3).
5. **Evaluasi dan visualisasi.** Node `Silhouette Coefficient` menghitung skor tiap hasil, dan node `Scatter Plot` menampilkan persebaran data beserta cluster-nya.

Workflow KNIME ini **tidak memakai node `Normalizer`**, sehingga data diproses langsung tanpa tahap normalisasi terpisah.

> Catatan: karena tanpa normalisasi, hasil K-Means di KNIME tidak selalu sama dengan bagian Python (yang memakai `StandardScaler`). Fitur berskala besar akan mendominasi jarak.

## 2.2 Workflow Linear (2 k-Means, K = 2)

```
                  ┌→ k-Means (K=2) ───────────────────────→ Scatter Plot (204 fitur)
DB Reader (Linear)┤
                  └→ Python Script → k-Means (K=2) → Scatter Plot (203/74/37 fitur)
```

1. **k-Means tanpa reduksi dimensi.** Data langsung masuk ke `k-Means` (K = 2) dengan 204 fitur.
2. **k-Means dengan reduksi dimensi.** Data melewati `Python Script` (reduksi ke 203, 74, atau 37 fitur), lalu masuk ke `k-Means` (K = 2). Jumlah fitur target diganti pada node `Python Script` untuk tiap kondisi.

## 2.3 Workflow Polynomial (3 k-Means, K = 2 dan K = 3)

```
                      ┌→ k-Means (K=2) ───────────────────────→ Scatter Plot (204 fitur)
DB Reader (Polynomial)┤
                      └→ Python Script ─┬→ k-Means (K=2) → Scatter Plot (203/74 fitur)
                                        └→ k-Means (K=3) → Scatter Plot (37 fitur)
```

1. **k-Means tanpa reduksi dimensi, K = 2.** Data langsung masuk ke `k-Means` dengan 204 fitur.
2. **k-Means dengan reduksi dimensi 203 dan 74 fitur, K = 2.** Data melewati `Python Script` dengan target 203, lalu 74 fitur, kemudian masuk ke `k-Means` (K = 2).
3. **k-Means dengan reduksi dimensi 37 fitur, K = 3.** Data melewati `Python Script` dengan target 37 fitur, lalu masuk ke `k-Means` (K = 3).

## 2.4 Konfigurasi Node Python Script (Reduksi Dimensi)

Reduksi dimensi memakai `FeatureAgglomeration` dengan `linkage="ward"`, sama dengan bagian Python. Node ini mengelompokkan kolom fitur yang mirip, lalu mengganti tiap kelompok dengan rata-rata anggotanya, sehingga jumlah kolom akhir tepat sesuai target (203, 74, atau 37). Kolom teks (`nama`, `daerah`) dibiarkan dan ikut ke output.

Kode berikut dijalankan **di dalam node Python Script KNIME**, bukan di notebook ini (modul `knime.scripting.io` hanya ada di KNIME):

```python
import knime.scripting.io as knio
import pandas as pd
from sklearn.cluster import FeatureAgglomeration

N_FITUR = 37   # ganti: 203, 74, atau 37

df = knio.input_tables[0].to_pandas()

# pisahkan kolom teks (nama, daerah) dari kolom fitur numerik
kolom_teks = df.select_dtypes(exclude="number").columns.tolist()
kolom_fitur = df.select_dtypes(include="number").columns.tolist()

X = df[kolom_fitur].values
reducer = FeatureAgglomeration(n_clusters=N_FITUR, linkage="ward")
X_reduksi = reducer.fit_transform(X)

df_fitur = pd.DataFrame(
    X_reduksi,
    index=df.index,
    columns=[f"fitur_{i+1}" for i in range(N_FITUR)],
)

hasil = pd.concat([df[kolom_teks], df_fitur], axis=1)
knio.output_tables[0] = knio.Table.from_pandas(hasil)
```

**Penjelasan kode:**
1. `knime.scripting.io` adalah modul KNIME untuk membaca tabel masukan dan menulis tabel keluaran. `N_FITUR` menentukan jumlah kolom akhir dan diganti untuk tiap kondisi.
2. `knio.input_tables[0].to_pandas()` mengambil tabel dari port masukan node sebagai DataFrame.
3. `select_dtypes(exclude="number")` mengambil kolom teks (`nama`, `daerah`), sedangkan `select_dtypes(include="number")` mengambil kolom fitur numerik. **Kolom `id` harus dikeluarkan sebelum node ini**, kalau tidak `id` dianggap fitur.
4. `FeatureAgglomeration(...).fit_transform(X)` mengelompokkan kolom menjadi `N_FITUR` kelompok dan mengganti tiap kelompok dengan rata-ratanya.
5. Matriks hasil diubah menjadi DataFrame berkolom `fitur_1` sampai `fitur_N`, lalu digabung ke samping dengan kolom teks.
6. `knio.output_tables[0] = ...` mengirim tabel hasil ke port keluaran node, lalu diteruskan ke `k-Means`.

## 2.5 Hasil Implementasi

Skenario **Linear** menghasilkan empat hasil clustering (204, 203, 74, dan 37 fitur) dengan K = 2. Skenario **Polynomial** menghasilkan empat hasil dengan K = 2 untuk 204, 203, dan 74 fitur, serta K = 3 untuk 37 fitur. Hasil Scatter Plot digunakan sebagai visualisasi pendukung untuk melihat persebaran data dan pembentukan cluster.

Sebagai pembanding, pada bagian Python dengan K = 2, cluster kecil Linear berisi tiga daerah (Jabon dan Wonoayu di Sidoarjo, serta Tanah Merah di Bangkalan), sedangkan cluster kecil Polynomial berisi dua daerah (Jabon dan Wonoayu). Pada Polynomial 37 fitur dengan K = 3, Kamal (Muhammad Zaidan Nabil Rafi) dan Jabon masing-masing membentuk cluster sendiri.